In [0]:
CREATE OR REFRESH STREAMING TABLE mavenmarket.gold.fact_inventory
CLUSTER BY (restock_date_key, store_sk)
COMMENT 'Store Inventory Restocking Events Fact Table'
AS
SELECT 
    -- Primary Business Key / Event ID
    TRIM(i.event_id) AS event_id,
    
    -- Date Foreign Key (mapped directly to dim_date.date_sk in YYYYMMDD format)
    COALESCE(
        d.date_sk, 
        CAST(date_format(i.restock_date, 'yyyyMMdd') AS INT)
    ) AS restock_date_key,
    
    -- Direct Surrogate Key mappings from Silver
    i.product_sk,
    i.store_sk,
    
    COALESCE(i.restock_quantity, 0) AS restock_quantity

FROM STREAM(mavenmarket.silver.silver_inventory) i

-- Stream-to-Batch Join with Gold Date Dimension using calendar_date
LEFT JOIN mavenmarket.gold.dim_date d 
    ON i.restock_date = d.calendar_date

WHERE i.restock_date IS NOT NULL;